# Filtered-NF policy trajectories

This notebook restores the critic-filtered trajectory flow and the continued PPO policy, samples a batch of target motions, and plots the robot's realized x–y trajectories.

Rollout points at and after the first underlying environment auto-reset are masked, so a fallen agent does not produce a spurious line back to the reset position. Actions are deterministic by default for a clean policy visualization.

In [ ]:
from pathlib import Path
import sys

import flax.linen as nn
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from brax import envs
from flax import serialization

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "jax_rq_nsf").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from jax_rq_nsf import FlowConfig, init_flow
from networks import PPO_Policy
from task_wrappers.nf_trajectory_following import AntFiniteMaternWrapper

FLOW_CHECKPOINT = PROJECT_ROOT / "output/vi_filtered/trajectory_flow_variables.msgpack"
POLICY_CHECKPOINT = PROJECT_ROOT / "output/vi_filtered_ppo/saved_data/policy.msgpack"
FIGURE_DIRECTORY = PROJECT_ROOT / "output/visualizations"

SEED = 8848
NUM_TRAJECTORIES = 128
NUM_DETAIL_PLOTS = 12
STOCHASTIC_ACTIONS = False

WAY_POINTS = 12
STEPS_PER_WAY_POINT = 8
DT = 0.05
FLOW_CONFIG = FlowConfig(
    dimension=4 * WAY_POINTS + 2,
    num_layers=8,
    hidden_features=(128, 128),
    num_bins=16,
    tail_bound=6.0,
)

assert FLOW_CHECKPOINT.exists(), FLOW_CHECKPOINT
assert POLICY_CHECKPOINT.exists(), POLICY_CHECKPOINT
FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
print("JAX devices:", jax.devices())

In [ ]:
# Restore the complete flow variables tree.
flow, flow_variables_template = init_flow(jax.random.PRNGKey(0), FLOW_CONFIG)
flow_variables = serialization.from_bytes(
    flow_variables_template,
    FLOW_CHECKPOINT.read_bytes(),
)

base_env = envs.create(
    env_name="ant",
    episode_length=4096,
    backend="mjx",
    reset_noise_scale=0.0,
)
env = AntFiniteMaternWrapper(
    base_env,
    flow,
    way_points=WAY_POINTS,
    steps_per_way_point=STEPS_PER_WAY_POINT,
    dt=DT,
    inner_radius=0.5,
    max_radius=2.0,
    tolerance_radius=0.1,
)

policy_network = PPO_Policy(
    hidden_layer_sizes=(256, 256),
    action_dim=env.action_size,
    kernel_init=jax.nn.initializers.orthogonal(jnp.sqrt(2.0)),
    kernel_init_final=jax.nn.initializers.orthogonal(0.01),
    activation=nn.silu,
    final_activation=jnp.tanh,
)
policy_template = policy_network.init(
    jax.random.PRNGKey(1),
    obs=jnp.zeros((env.observation_size,)),
    z=jnp.zeros((env.z_size,)),
)
policy_variables = serialization.from_bytes(
    policy_template,
    POLICY_CHECKPOINT.read_bytes(),
)

print("Flow checkpoint:", FLOW_CHECKPOINT)
print("Policy checkpoint:", POLICY_CHECKPOINT)
print("Observation/action/z dimensions:", env.observation_size, env.action_size, env.z_size)

In [ ]:
# Reset one environment per sampled task. Reproduce BaseFlowQDWrapper's key split
# so flow_samples is exactly the task vector stored in each initial z[5:].
master_key = jax.random.PRNGKey(SEED)
master_key, reset_key, action_key = jax.random.split(master_key, 3)
reset_keys = jax.random.split(reset_key, NUM_TRAJECTORIES)
reset_key_parts = jax.vmap(lambda key: jax.random.split(key, 3))(reset_keys)
task_keys = reset_key_parts[:, 1]

flow_samples = jax.vmap(
    lambda key: flow.apply(
        flow_variables,
        key,
        1,
        method=flow.sample,
    )[0]
)(task_keys)

initial_states = jax.vmap(env.reset, in_axes=(0, None))(
    reset_keys,
    flow_variables,
)
max_task_mismatch = jnp.max(jnp.abs(initial_states.z_state.z[:, 5:] - flow_samples))
print("Maximum sampled-task mismatch:", float(max_task_mismatch))


def target_step(task_state, _):
    cycle_t = task_state.cycle_t + env.dt
    base_values = env._base_fn(cycle_t)
    shifted_positions = jnp.sum(
        (task_state.reshaped_sequence @ env.Sigma22_inv_S) * base_values,
        axis=-1,
    )
    return task_state.replace(cycle_t=cycle_t), shifted_positions[:, 0]


def target_period(task_state, _):
    task_state, targets = jax.lax.scan(
        target_step,
        task_state,
        xs=None,
        length=env.steps_per_way_point,
    )
    shifted_sequence = jnp.concatenate(
        [task_state.reshaped_sequence[:, 1:, :], task_state.padding_element],
        axis=1,
    )
    task_state = task_state.replace(
        cycle_t=task_state.cycle_t - env.period_t,
        reshaped_sequence=shifted_sequence,
    )
    return task_state, targets


def sample_smooth_target(task_state):
    _, target_blocks = jax.lax.scan(
        target_period,
        task_state,
        xs=None,
        length=env.way_points,
    )
    return target_blocks.reshape(env.max_step_num, 2)


smooth_targets = jax.jit(jax.vmap(sample_smooth_target))(initial_states.z_state)
smooth_targets = jnp.concatenate(
    [jnp.zeros((NUM_TRAJECTORIES, 1, 2)), smooth_targets],
    axis=1,
)
print("Sampled flow tasks:", flow_samples.shape)
print("Smooth target paths:", smooth_targets.shape)

In [ ]:
def rollout_step(carry, _):
    states, alive, key = carry
    observations, zs = env.get_obs(states)
    action_mean, std_logits = policy_network.apply(
        policy_variables,
        observations,
        zs,
    )

    key, noise_key = jax.random.split(key)
    if STOCHASTIC_ACTIONS:
        action_std = nn.sigmoid(std_logits)
        actions = action_mean + action_std * jax.random.normal(
            noise_key,
            action_mean.shape,
        )
    else:
        actions = action_mean

    next_states, _ = jax.vmap(env.step)(
        states,
        jnp.clip(actions, -1.0, 1.0),
    )
    auto_reset = next_states.env_state.done > 0.5
    valid = alive & ~auto_reset
    positions = next_states.env_state.pipeline_state.x.pos[:, 0, :2]
    return (next_states, valid, key), (positions, valid, auto_reset)


def rollout_period(carry, _):
    carry, step_data = jax.lax.scan(
        rollout_step,
        carry,
        xs=None,
        length=env.steps_per_way_point,
    )
    states, alive, key = carry
    states = jax.vmap(env.shift)(states)
    return (states, alive, key), step_data


@jax.jit
def rollout_batch(states, key):
    initial_alive = jnp.ones((NUM_TRAJECTORIES,), dtype=jnp.bool_)
    final_carry, rollout_blocks = jax.lax.scan(
        rollout_period,
        (states, initial_alive, key),
        xs=None,
        length=env.way_points,
    )
    return final_carry, rollout_blocks


initial_positions = initial_states.env_state.pipeline_state.x.pos[:, 0, :2]
(_, final_alive, _), (position_blocks, valid_blocks, reset_blocks) = rollout_batch(
    initial_states,
    action_key,
)

# Convert (waypoint, step, batch, ...) to (batch, time, ...), and express
# positions relative to each rollout's initial x-y location.
rollout_positions = position_blocks.reshape(env.max_step_num, NUM_TRAJECTORIES, 2)
rollout_positions = jnp.swapaxes(rollout_positions, 0, 1)
rollout_positions = rollout_positions - initial_positions[:, None, :]
rollout_positions = jnp.concatenate(
    [jnp.zeros((NUM_TRAJECTORIES, 1, 2)), rollout_positions],
    axis=1,
)

valid_masks = valid_blocks.reshape(env.max_step_num, NUM_TRAJECTORIES).T
valid_masks = jnp.concatenate(
    [jnp.ones((NUM_TRAJECTORIES, 1), dtype=jnp.bool_), valid_masks],
    axis=1,
)
reset_masks = reset_blocks.reshape(env.max_step_num, NUM_TRAJECTORIES).T

rollout_positions = np.asarray(rollout_positions)
smooth_targets = np.asarray(smooth_targets)
valid_masks = np.asarray(valid_masks)
reset_masks = np.asarray(reset_masks)

survival_steps = valid_masks.sum(axis=1) - 1
print(f"Completed without auto-reset: {int(np.asarray(final_alive).sum())}/{NUM_TRAJECTORIES}")
print("Survival steps: min / mean / max =", survival_steps.min(), survival_steps.mean(), survival_steps.max())

In [ ]:
# Directly comparable policy-diversity plot: one realized x-y path per NF task.
colors = plt.cm.turbo(np.linspace(0.0, 1.0, NUM_TRAJECTORIES))
fig, ax = plt.subplots(figsize=(9, 9), dpi=140)

for index, color in enumerate(colors):
    path = rollout_positions[index, valid_masks[index]]
    ax.plot(path[:, 0], path[:, 1], color=color, linewidth=1.6, alpha=0.9)
    ax.scatter(path[-1, 0], path[-1, 1], color=color, s=10)

ax.scatter(0.0, 0.0, color="black", marker="x", s=50, label="start", zorder=5)
ax.set_title(
    f"Filtered-NF conditioned policy: {NUM_TRAJECTORIES} sampled tasks"
    + (" (stochastic actions)" if STOCHASTIC_ACTIONS else " (deterministic actions)")
)
ax.set_xlabel("relative x position")
ax.set_ylabel("relative y position")
ax.set_aspect("equal", adjustable="datalim")
ax.grid(alpha=0.2)
ax.legend(loc="best")
fig.tight_layout()

policy_figure_path = FIGURE_DIRECTORY / "filtered_nf_policy_xy.png"
fig.savefig(policy_figure_path, bbox_inches="tight")
plt.show()
print("Saved:", policy_figure_path)

In [ ]:
# Tracking diagnostics: dashed target path versus solid realized path.
num_panels = min(NUM_DETAIL_PLOTS, NUM_TRAJECTORIES)
num_columns = 4
num_rows = int(np.ceil(num_panels / num_columns))
fig, axes = plt.subplots(
    num_rows,
    num_columns,
    figsize=(4 * num_columns, 4 * num_rows),
    dpi=120,
    squeeze=False,
)

for index, ax in enumerate(axes.flat):
    if index >= num_panels:
        ax.axis("off")
        continue

    target = smooth_targets[index]
    actual = rollout_positions[index, valid_masks[index]]
    ax.plot(
        target[:, 0],
        target[:, 1],
        "--",
        color="black",
        linewidth=1.3,
        alpha=0.65,
        label="NF target",
    )
    ax.plot(
        actual[:, 0],
        actual[:, 1],
        color=colors[index],
        linewidth=2.0,
        label="robot",
    )
    ax.scatter(0.0, 0.0, color="black", marker="x", s=25)
    ax.scatter(actual[-1, 0], actual[-1, 1], color=colors[index], s=18)
    status = "complete" if survival_steps[index] == env.max_step_num else f"reset at {survival_steps[index]}"
    ax.set_title(f"Task {index}: {status}")
    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(alpha=0.2)
    if index == 0:
        ax.legend(loc="best")

fig.suptitle("Sampled NF targets and policy rollouts", y=1.01)
fig.tight_layout()
detail_figure_path = FIGURE_DIRECTORY / "filtered_nf_target_vs_policy.png"
fig.savefig(detail_figure_path, bbox_inches="tight")
plt.show()
print("Saved:", detail_figure_path)